# Lecture 6 — Class Exercise
## Choropleth Maps

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px.choropleth` or `px.choropleth_map` — choose deliberately and state your reason
2. Right colour scale for your data (sequential vs diverging) — state which and why
3. Insight title names a geographic finding — not just a topic
4. `featureidkey` must be correctly matched to your GeoJSON

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json


## Task 1 — World choropleth: life expectancy diverging scale

**What to build:** A world choropleth showing **life expectancy relative to the global average** using a diverging colour scale.

**Requirements:**
- Use the Gapminder dataset for 2007: `px.data.gapminder()`
- Compute each country's deviation from the global mean life expectancy
- Diverging scale centred at zero (= world average)
- `hover_data` showing country name, raw life expectancy, and deviation
- Insight title naming which region is furthest below average

> 💡 `gm_2007['lifeExp'].mean()` gives you the global average to subtract from


In [3]:

# Task 1 - World choropleth

import plotly.express as px
import pandas as pd

# Load Gapminder dataset for 2007
gm_2007 = px.data.gapminder().query("year == 2007").copy()

# Calculate global average life expectancy
global_avg = gm_2007['lifeExp'].mean()

# Calculate deviation from global average
gm_2007['deviation'] = gm_2007['lifeExp'] - global_avg

# Find the region with the lowest average life expectancy
region_avg = gm_2007.groupby('continent')['lifeExp'].mean()

lowest_region = region_avg.idxmin()

# Create insight title
title = f"{lowest_region} Has the Lowest Life Expectancy Compared with the Global Average"

# Create world choropleth map
fig = px.choropleth(
    gm_2007,
    locations='iso_alpha',
    color='deviation',
    hover_name='country',
    hover_data={
        'lifeExp': ':.2f',
        'deviation': ':+.2f',
        'iso_alpha': False
    },
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    title=title,
    labels={
        'deviation': 'Difference from Global Average (Years)',
        'lifeExp': 'Life Expectancy'
    }
)

fig.update_layout(
    template='plotly_white',
    title_x=0.5
)

fig.show()

print("Global Average Life Expectancy:", round(global_avg, 2))
print("Region furthest below average:", lowest_region)


Global Average Life Expectancy: 67.01
Region furthest below average: Africa


## Task 2 — Find your own GeoJSON

**What to build:** A choropleth using a GeoJSON file you find yourself online.

**Requirements:**
- Find a free GeoJSON file for any geography that interests you (country, region, city)
- Create or find a matching dataset with at least one numeric variable per region
- Build either a `px.choropleth` or `px.choropleth_mapbox` — state your choice and reason in the markdown cell below
- Correctly identify and set `featureidkey` by inspecting the GeoJSON properties
- Choose sequential or diverging scale — state your reason in the markdown cell below
- Insight title naming a geographic finding

**Where to find GeoJSON files:**
- [geojson.xyz](https://geojson.xyz/) — countries, cities, natural features
- [naturalearthdata.com](https://www.naturalearthdata.com/) — global admin boundaries
- [github.com/datasets/geo-countries](https://github.com/datasets/geo-countries) — country polygons
- Search: `[country name] [admin level] GeoJSON github` — most countries have free boundary files on GitHub

> 💡 Before plotting, always inspect your GeoJSON properties first:
> ```python
> print(my_geojson['features'][0]['properties'])
> ```
> The property name that matches your dataframe's location column is what goes in `featureidkey='properties.???'`


### Task 2 — Design decisions

**GeoJSON source:** 
https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson

**Chart type chosen:** px.choropleth

Reason: It provides a clear world map showing life expectancy across different countries.

**Colour scale chosen:** Sequential (Viridis)

Reason: Life expectancy is a numerical value, so a sequential colour scale makes it easy to compare countries.

In [ ]:

# Task 2 - Find Your Own GeoJSON
# World Life Expectancy Choropleth Map

import pandas as pd
import plotly.express as px
import requests
import json

# Step 1: Download the World GeoJSON file
geojson_url = "https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson"

response = requests.get(geojson_url)
response.raise_for_status()

my_geojson = response.json()

# Step 2: Inspect GeoJSON properties
print("GeoJSON properties:")
print(my_geojson['features'][0]['properties'])

# Step 3: Load Gapminder dataset for 2007
gm_2007 = px.data.gapminder().query("year == 2007").copy()

# Step 4: Find highest and lowest life expectancy
highest = gm_2007.loc[gm_2007['lifeExp'].idxmax()]
lowest = gm_2007.loc[gm_2007['lifeExp'].idxmin()]

# Step 5: Calculate global average
global_avg = gm_2007['lifeExp'].mean()

# Step 6: Create insight title
title = (
    f"{highest['country']} Has the Highest Life Expectancy, "
    f"While {lowest['country']} Has the Lowest"
)

# Step 7: Create choropleth map
fig = px.choropleth(
    gm_2007,
    geojson=my_geojson,
    locations='iso_alpha',
    featureidkey='properties.ISO3166-1-Alpha-3',
    color='lifeExp',
    hover_name='country',
    hover_data={
        'lifeExp': ':.2f',
        'iso_alpha': False
    },
    color_continuous_scale='Viridis',
    title=title,
    labels={
        'lifeExp': 'Life Expectancy (Years)'
    }
)

# Step 8: Format map
fig.update_geos(
    fitbounds="locations",
    visible=False
)

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    height=600
)

# Step 9: Display map
fig.show()

# Step 10: Print findings
print("Global Average Life Expectancy:", round(global_avg, 2))
print("Highest life expectancy:", highest['country'], round(highest['lifeExp'], 2))
print("Lowest life expectancy:", lowest['country'], round(lowest['lifeExp'], 2))
